# Lab 15.4: Asking the Right Question of a Representation

Run every cell in order. Nothing needs to be edited, and no language model is needed: everything is synthetic and runs in seconds. Two experiments:

1. **Mine or theirs.** A synthetic "board" is stored in a vector relative to the player about to move, the way Nanda, Lee and Wattenberg (2023) found Othello-GPT stores it. We probe it with two labelings of the same squares.
2. **Decodable but not used.** A small network learns a task; we probe its hidden layer for a property the task needs and one it ignores, then remove each property (amnesic probing in the spirit of Elazar et al., 2021) and see which removal changes the network's behavior.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
rng = np.random.default_rng(0)
N, T, D = 6000, 16, 64
rel = rng.integers(0, 3, size=(N, T))          # 0 empty, 1 mine, 2 theirs (relative to the player to move)
p = rng.integers(0, 2, size=N)                 # whose turn: 0 black, 1 white
color = np.where(rel == 0, 0, np.where((rel == 1) == (p[:, None] == 0), 1, 2))  # 1 black, 2 white
Em, Et, pd_dir = rng.normal(size=(T, D)), rng.normal(size=(T, D)), rng.normal(size=D)
H = (rel == 1) @ Em + (rel == 2) @ Et + np.outer(p, pd_dir) + rng.normal(size=(N, D))
tr, te = slice(0, 4000), slice(4000, N)
rows = []
for t in range(8):
    rows.append({"square": t,
                 "linear, black/white/empty": LogisticRegression(max_iter=3000).fit(H[tr], color[tr, t]).score(H[te], color[te, t]),
                 "MLP, black/white/empty": MLPClassifier((64,), max_iter=1500, random_state=0).fit(H[tr], color[tr, t]).score(H[te], color[te, t]),
                 "linear, mine/theirs/empty": LogisticRegression(max_iter=3000).fit(H[tr], rel[tr, t]).score(H[te], rel[te, t])})
board = pd.DataFrame(rows).set_index("square")
display(board.round(3)); print(board.mean().round(3))

,"linear, black/white/empty","MLP, black/white/empty","linear, mine/theirs/empty"
square,,,
0,0.654,0.990,0.996
1,0.657,0.992,0.998
2,0.660,0.992,0.998
3,0.664,0.980,0.997
4,0.652,0.975,0.995
5,0.659,0.976,0.988
6,0.655,0.988,0.998
7,0.648,0.980,0.994


linear, black/white/empty    0.656
MLP, black/white/empty       0.984
linear, mine/theirs/empty    0.995
dtype: float64


**Checkpoint 1.** Mean accuracy over eight squares: linear probe with black/white labels about 0.66, a nonlinear (MLP) probe about 0.98, linear probe with mine/theirs labels about 1.00. The information is the same in all three columns; only the question changed. Why can a linear probe not read black or white here? (Hint: black = mine when black is to move, and theirs otherwise.)

In [2]:
N, D = 8000, 20
X = rng.normal(size=(N, D))
y = (X[:, 0] + X[:, 1] > 0).astype(int)        # the task
b = (X[:, 5] > 0).astype(int)                  # a property the task ignores
net = MLPClassifier((32,), max_iter=2000, random_state=0).fit(X[:5000], y[:5000])
W1, W2 = net.coefs_; b1, b2 = net.intercepts_
Hh = np.maximum(0, X @ W1 + b1)                # the hidden layer we probe
tr, te = slice(0, 5000), slice(5000, N)
task = lambda Hm: (((Hm @ W2 + b2).ravel() > 0) == y)[te].mean()
mu = Hh[tr].mean(0)
print("network's task accuracy:", round(task(Hh), 3))
out = []
for name, lab in (("ignored property", b), ("needed property", y)):
    Hc = Hh - mu
    before = LogisticRegression(max_iter=3000).fit(Hh[tr], lab[tr]).score(Hh[te], lab[te])
    for _ in range(8):                         # remove the probe direction, refit, repeat
        w = LogisticRegression(max_iter=3000).fit(Hc[tr], lab[tr]).coef_.ravel(); w /= np.linalg.norm(w)
        Hc = Hc - np.outer(Hc @ w, w)
    after = LogisticRegression(max_iter=3000).fit(Hc[tr], lab[tr]).score(Hc[te], lab[te])
    out.append({"property": name, "probe accuracy": before, "probe accuracy after removal": after,
                "task accuracy after removal": task(Hc + mu)})
pd.DataFrame(out).set_index("property").round(3)

network's task accuracy: 0.99


,probe accuracy,probe accuracy after removal,task accuracy after removal
property,,,
ignored property,0.843,0.540,0.990
needed property,0.991,0.631,0.488


**Checkpoint 2.** Both properties are decodable (probe accuracy about 0.84 for the ignored one, about 0.99 for the needed one). Removing the ignored property leaves the network's task accuracy at about 0.99; removing the needed one drops it to about 0.49, which is chance. Decodable is not the same as used.